In [31]:
import pandas as pd

In [94]:
# read data files 
orders = pd.read_csv("../data/orders.csv")
customers = pd.read_csv("../data/customers.csv")
products = pd.read_csv("../data/products.csv")
sellers = pd.read_csv("../data/sellers.csv")
order_payments = pd.read_csv("../data/order_payments.csv")
order_items = pd.read_csv("../data/order_items.csv")
order_reviews = pd.read_csv("../data/order_reviews.csv")

In [ ]:
# cleaning orders dataframme 
orders.isnull().sum()
orders["order_status"].value_counts()
orders_delivered = orders[orders["order_status"] == "delivered"]
orders_delivered.isnull().sum()
date_columns = [
"order_purchase_timestamp",
"order_approved_at",
"order_delivered_carrier_date",
"order_delivered_customer_date",
"order_estimated_delivery_date"
]
for col in date_columns:
    orders_delivered[col] = pd.to_datetime(orders_delivered[col])

orders_delivered.duplicated().sum()
orders_delivered = orders_delivered.drop_duplicates()


In [ ]:
# cleaning order_items dataframme 
order_items.isnull().sum()
order_items.duplicated().sum()
order_items["freight_value"] = order_items["freight_value"].clip(lower=0)
order_items = order_items[order_items["order_id"].isin(orders["order_id"])]
order_items.head()
order_items["order_item_id"] = order_items["order_item_id"].astype(int)
order_items["price"] = order_items["price"].astype(float)
order_items["freight_value"] = order_items["freight_value"].astype(float)

In [92]:
#cleaning products dataframme 
products.isnull().sum()
products['product_category_name'] = products['product_category_name'].fillna("unknown")
products[products.duplicated(subset=["product_id"])]
products = products.drop_duplicates(subset=["product_id"])
numeric_cols = ["product_name_lenght","product_description_lenght","product_photos_qty",
                "product_weight_g","product_length_cm","product_height_cm","product_width_cm"]

for col in numeric_cols:
    products[col] = pd.to_numeric(products[col], errors='coerce')

products = products[(products["product_weight_g"] > 0) & 
                    (products["product_length_cm"] > 0) & 
                    (products["product_height_cm"] > 0) &
                    (products["product_width_cm"] > 0)]

In [ ]:
#cleaning customers dataframme 
customers.isnull().sum()
customers = customers.dropna(subset=["customer_id","customer_unique_id"])
customers['customer_city'] = customers['customer_city'].fillna("unknown")
customers['customer_state'] = customers['customer_state'].fillna("unknown")
customers[customers.duplicated(subset=["customer_id"])]
customers = customers.drop_duplicates(subset=["customer_id"])
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str)
customers = customers[customers["customer_id"].isin(orders["customer_id"])]

In [ ]:
# cleaning payments dataframme 
order_payments.isnull().sum()
order_payments = order_payments.dropna(subset=["order_id","payment_value"])
order_payments['payment_installments'] = order_payments['payment_installments'].fillna(1)
order_payments[order_payments.duplicated(subset=["order_id","payment_sequential"])]
order_payments = order_payments.drop_duplicates(subset=["order_id","payment_sequential"])
order_payments = order_payments[order_payments["payment_value"] > 0]
order_payments["payment_installments"] = order_payments["payment_installments"].astype(int)
order_payments = order_payments[order_payments["payment_installments"] > 0]
order_payments = order_payments[order_payments["order_id"].isin(orders["order_id"])]